# TM-in-KS: Transition Matrix Inside the Krusell-Smith Loop

This notebook demonstrates using the transition-matrix method (via
`make_history_tm()`) as an alternative to Monte Carlo simulation inside
the Krusell-Smith aggregate equilibrium iteration.

We compare:
1. **MC-KS**: The standard HARK approach — solve agents, simulate via MC, regress to update AFunc
2. **TM forward propagation**: Using the converged AFunc from MC-KS, propagate a distribution via TM

Both methods use `CobbDouglasMarkovEconomy` with a 2-state boom/recession model.


In [1]:
import time
import numpy as np
from HARK.ConsumptionSaving.ConsAggShockModel import (
    AggShockMarkovConsumerType,
    CobbDouglasMarkovEconomy,
)

## 1. Solve economy with standard MC-KS

In [2]:
consumer = AggShockMarkovConsumerType()
consumer.cycles = 0

econ = CobbDouglasMarkovEconomy(agents=[consumer], verbose=True)
econ.make_AggShkHist()
econ.give_agent_params()

print(f"MrkvArray:\n{econ.MrkvArray}")
print(f"act_T = {econ.act_T}")
J = econ.MrkvArray.shape[0]
print(f"Markov state counts: {[np.sum(econ.MrkvNow_hist == j) for j in range(J)]}")

t0 = time.time()
econ.solve()
mc_time = time.time() - t0
print(f"\nMC-KS solved in {mc_time:.1f} seconds")
print(f"Converged AFunc intercepts: {econ.intercept_prev}")
print(f"Converged AFunc slopes: {econ.slope_prev}")

MrkvArray:
[[0.9  0.1 ]
 [0.04 0.96]]
act_T = 1200
Markov state counts: [np.int64(381), np.int64(819)]


intercept=[np.float64(-0.482916418198766), np.float64(-0.6286768645335784)], slope=[np.float64(1.1228671690799255), np.float64(1.1975925224299746)], r-sq=[np.float64(0.9983566435141913), np.float64(0.993999530186829)]


intercept=[np.float64(-0.31136898110022687), np.float64(-0.3166362223917491)], slope=[np.float64(1.0475161827021613), np.float64(1.0423783867707275)], r-sq=[np.float64(0.9998795814756987), np.float64(0.9997235487802072)]


intercept=[np.float64(-0.31772821019139563), np.float64(-0.3607467625500071)], slope=[np.float64(1.060032867137027), np.float64(1.0716972339492286)], r-sq=[np.float64(0.999947296464975), np.float64(0.9999330431328163)]


intercept=[np.float64(-0.33676608074401204), np.float64(-0.3838533140845089)], slope=[np.float64(1.066431108810626), np.float64(1.0798799983131717)], r-sq=[np.float64(0.9999431892865129), np.float64(0.9999287885763215)]


intercept=[np.float64(-0.3336881186128549), np.float64(-0.37898863566455954)], slope=[np.float64(1.0653641129930684), np.float64(1.0782096210498118)], r-sq=[np.float64(0.9999437881173592), np.float64(0.999928767185237)]


intercept=[np.float64(-0.3343143343570017), np.float64(-0.38001751642276044)], slope=[np.float64(1.0655782892719032), np.float64(1.078561138667419)], r-sq=[np.float64(0.9999437446206528), np.float64(0.9999288287833763)]


intercept=[np.float64(-0.3341768589228499), np.float64(-0.3797942743282447)], slope=[np.float64(1.06553141782149), np.float64(1.0784848717846809)], r-sq=[np.float64(0.9999437569948211), np.float64(0.9999288208033832)]


intercept=[np.float64(-0.334206890398842), np.float64(-0.379842837443582)], slope=[np.float64(1.0655416565055962), np.float64(1.078501460303492)], r-sq=[np.float64(0.9999437542500996), np.float64(0.9999288225499203)]

MC-KS solved in 243.4 seconds
Converged AFunc intercepts: [np.float64(-0.334206890398842), np.float64(-0.379842837443582)]
Converged AFunc slopes: [np.float64(1.0655416565055962), np.float64(1.078501460303492)]


## 2. Save MC history, then run TM forward propagation

In [3]:
# Save MC history (make_history_tm will overwrite econ.history)
mc_M = np.array(econ.history["MaggNow"]).copy()
mc_A = np.array(econ.history["AaggNow"]).copy()

t0 = time.time()
econ.make_history_tm(num_pointsM=200, mMax=50)
tm_time = time.time() - t0
print(f"TM forward propagation in {tm_time:.1f} seconds")
print(f"  (MC-KS took {mc_time:.1f} seconds total)")

tm_M = econ.history["MaggNow"].copy()
tm_A = econ.history["AaggNow"].copy()

TM forward propagation in 2.5 seconds
  (MC-KS took 243.4 seconds total)


## 3. Compare MC and TM aggregate trajectories

In [4]:
T_discard = econ.T_discard
T = min(len(mc_M), len(tm_M))
mc_M_trim = mc_M[T_discard:T]
mc_A_trim = mc_A[T_discard:T]
tm_M_trim = tm_M[T_discard:T]
tm_A_trim = tm_A[T_discard:T]

print(f"MC aggregate M: mean={mc_M_trim.mean():.4f}, std={mc_M_trim.std():.4f}")
print(f"TM aggregate M: mean={tm_M_trim.mean():.4f}, std={tm_M_trim.std():.4f}")
print(f"MC aggregate A: mean={mc_A_trim.mean():.4f}, std={mc_A_trim.std():.4f}")
print(f"TM aggregate A: mean={tm_A_trim.mean():.4f}, std={tm_A_trim.std():.4f}")

valid = (
    np.isfinite(tm_M_trim) & np.isfinite(mc_M_trim) & (tm_M_trim > 0) & (mc_M_trim > 0)
)
if np.sum(valid) > 10:
    corr_M = np.corrcoef(mc_M_trim[valid], tm_M_trim[valid])[0, 1]
    corr_A = np.corrcoef(mc_A_trim[valid], tm_A_trim[valid])[0, 1]
    print(f"\nCorrelation (M): {corr_M:.4f}")
    print(f"Correlation (A): {corr_A:.4f}")
else:
    print(f"\nInsufficient valid data for correlation ({np.sum(valid)} valid points)")

MC aggregate M: mean=13.0041, std=3.8828
TM aggregate M: mean=10.1334, std=2.5680
MC aggregate A: mean=10.9596, std=3.5636
TM aggregate A: mean=8.2434, std=2.3106

Correlation (M): 0.9969
Correlation (A): 0.9954


## 4. Fit AFunc from TM history

In [5]:
from scipy import stats as sp_stats

logAagg_tm = np.log(np.maximum(tm_A_trim, 1e-10))
logMagg_tm = np.log(np.maximum(tm_M[T_discard - 1 : T - 1], 1e-10))
MrkvHist_tm = econ.MrkvNow_hist[T_discard - 1 : T - 1]

StateCount = econ.MrkvArray.shape[0]
print("AFunc comparison (MC-converged vs TM-fitted):")
print(
    f"{'State':<6} {'MC intercept':>14} {'TM intercept':>14} {'MC slope':>10} {'TM slope':>10} {'TM R²':>8}"
)
for j in range(StateCount):
    these = j == MrkvHist_tm
    n = np.sum(these)
    if n < 10:
        print(f"{j:<6}  insufficient data (n={n})")
        continue
    slope, intercept, r_value, _, _ = sp_stats.linregress(
        logMagg_tm[these], logAagg_tm[these]
    )
    print(
        f"{j:<6} {econ.intercept_prev[j]:>14.6f} {intercept:>14.6f} "
        f"{econ.slope_prev[j]:>10.6f} {slope:>10.6f} {r_value**2:>8.6f}"
    )

AFunc comparison (MC-converged vs TM-fitted):
State    MC intercept   TM intercept   MC slope   TM slope    TM R²
0           -0.334207      -0.232835   1.065542   1.024541 0.996453
1           -0.379843      -0.322092   1.078501   1.040250 0.998048


## 5. Summary

The `make_history_tm()` method forward-propagates a distribution via
transition matrices as an alternative to Monte Carlo in the KS loop.

Key properties:
- **Deterministic**: Zero sampling noise for given aggregate shock sequence.
- **Fast**: Building a 1D TM at each time step via numba is much faster
  than simulating thousands of agents.
- **Compatible**: Produces the same `history` dict as `make_history()`.
